## Open in Colab

1. **Upload this file:** [Google Colab](https://colab.research.google.com/) → **File → Upload notebook** → pick `notebooks/SFT_Colab.ipynb` from this repo.

   *Or*, after you push to GitHub, open:  
   `https://colab.research.google.com/github/inezaodon/nanochat-replica/blob/main/notebooks/SFT_Colab.ipynb`  
   (change user/repo if you forked.)

2. **Enable GPU:** **Runtime → Change runtime type → GPU**

3. **Run all cells** (Runtime → Run all).

I **cannot** sign into your Google account or run Colab for you from here; this notebook is the automated substitute.

# Supervised fine-tuning (SFT) — Google Colab

This notebook runs **SFT on a Colab GPU** using this repo’s `llm/sft_train.py`: Alpaca-style prompts, **loss only on response tokens**, optional **partial freeze** of early transformer blocks.

**First:** **Runtime → Change runtime type → GPU** (T4 or similar).

If the GitHub repo is **private**, skip `git clone` and upload a zip of the project, then `cd` into the folder.

In [ ]:
# --- GPU check ---
import subprocess, sys

subprocess.run(["nvidia-smi"], check=False)
try:
    import torch
    print("torch:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("device:", torch.cuda.get_device_name(0))
except ImportError:
    print("torch not installed yet — run the next cells.")

In [ ]:
# --- Clone repo (change URL if you use a fork) ---
import os
import subprocess

REPO = "https://github.com/inezaodon/nanochat-replica.git"
WORKDIR = "/content/nanochat-replica"

if not os.path.isdir(WORKDIR):
    subprocess.run(["git", "clone", REPO, WORKDIR], check=True)
else:
    print("Repo already present:", WORKDIR)

os.chdir(WORKDIR)
print("cwd:", os.getcwd())

In [ ]:
# --- Install Python deps (run from repo root) ---
%cd /content/nanochat-replica
!pip install -q -r requirements.txt

In [ ]:
# --- Download ND course datasets into data/course/ (skips the 50 MB openweb dump) ---
%cd /content/nanochat-replica
!python -m llm.fetch_course_datasets

### Optional: full Stanford Alpaca (~52k rows, larger download)

Uncomment the next line if you want the full JSON instead of the smaller course `alpaca_instruction_data.json`.

In [ ]:
# !python -m llm.fetch_course_datasets --fetch-hf-alpaca

In [ ]:
# --- SFT: distilgpt2 on course Alpaca JSON (good default on Colab T4) ---
# Tune --steps (more = longer) and --max_samples (0 = all rows in the JSON file).

%cd /content/nanochat-replica
!python -m llm.sft_train \
  --device cuda \
  --model distilgpt2 \
  --data data/course/alpaca_instruction_data.json \
  --max_samples 0 \
  --max_length 512 \
  --batch_size 4 \
  --steps 600 \
  --lr 5e-5 \
  --trainable_blocks 1 \
  --out_dir /content/sft-checkpoints/distilgpt2-course-alpaca

### After training

- Weights are under `--out_dir` (above: `/content/sft-checkpoints/...`).
- **Download** that folder via Colab’s file browser, or zip it:
  `!zip -r /content/sft-out.zip /content/sft-checkpoints/distilgpt2-course-alpaca`

To SFT on the **full** Alpaca file, fetch it (uncomment the optional cell), then point `--data` to `data/course/stanford_alpaca_data.json` and raise `--steps` (and consider `--model gpt2` if you have enough VRAM).